# 04 — Street View Segmentation

`POST /v1/streetview` segments ground-level imagery (buildings, roads, vegetation, sky…) at a chosen viewing angle.

**Plan:** Premium only.

### Reference: viewing angles and `back_view`

The streetview endpoint orients a virtual camera at the requested lat/lon and segments what it sees. Three controls shape the view:

| Argument | Unit | Meaning |
|---|---|---|
| `vertical_angle` | degrees | Camera pitch — `0` looks straight ahead, positive values tilt the view upward (toward sky/canopy), negative values tilt downward (toward pavement) |
| `horizontal_angle` | degrees | Camera yaw — rotates the forward direction. `0` is the panorama's native forward; `90` rotates ninety degrees to one side, etc. |
| `back_view` | bool | When `True`, the response also includes a `back` block (mirror of `front`) facing the opposite direction. The use-case notebooks all use `False`; the bundled cached responses contain only the `front` block. |

The notebooks across this quickstart use small upward pitches and either forward (`0°`) or sideways (`90°`) yaw — see what each notebook actually passes:

| Notebook | `vertical_angle` | `horizontal_angle` | `back_view` |
|---|---|---|---|
| `04_street_view_segmentation.ipynb` (this one) | `10.0` | `90.0` | `False` |
| `notebooks/use_cases/urban_planner_bus_stop_prioritization.ipynb` | `5.0` | `0.0` | `False` |

A small positive `vertical_angle` (5–10°) generally captures more of the sky and tree canopy, which is what the use-case notebooks need for shade-and-canopy analysis. Adjust the values when the goal changes — e.g. set `vertical_angle` slightly negative if you want to emphasize pavement coverage.

> **Coverage gaps.** The streetview endpoint draws on ground-level panorama imagery, which is dense in U.S. urban areas but can be sparse on private property, rural roads, or recently-built developments. A request at coordinates with no nearby panorama returns an empty or error response — handle this case in production code.

### Reference: response schema

`client.street_view_segmentation(...)` returns `{"activity_id": str, "result": dict}`. The `result` carries:

- **`coordinates`** — `{latitude, longitude}` (as strings in the bundled samples) echoed back.
- **`front`** — primary view block:
  - `original_image` — base64 of the unmodified ground-level photo.
  - `segmented_image` — base64 of the pixel-wise segmentation overlay.
  - `segments` — `{class_name: percent_coverage}` per class.
  - `image_legend` — `{class_name: hex_color}` mapping for rendering the legend.
  - `image_date` — when the underlying panorama was captured.
- **`back`** — only present when `back_view=True`; same shape as `front`.

### Reference: class vocabulary

The classes returned by the streetview model (illustrative — classes vary by location; this notebook queries New York):

| Class | Typical use |
|---|---|
| `sky` | Canopy gap / openness — higher `sky` → more direct sun on the viewer |
| `tree` | Ground-level shade signal — distinct from `tree` in satellite (which is overhead) |
| `building` | Walls visible from the street; can also shade or trap heat |
| `road`, `sidewalk` | Surrounding paved surfaces |
| `car` | Foreground traffic |
| `earth` | Exposed ground / bare soil |
| `others` | Uncategorized pixels |

The use-case notebooks group these into `tree / building / sky / road` buckets for the action-list scoring.


In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))
from dotenv import load_dotenv; load_dotenv(pathlib.Path.cwd().parent / '.env')

from fortyguard import FortyGuardClient
client = FortyGuardClient()

In [ ]:
response = client.street_view_segmentation(
    latitude=40.7128,
    longitude=-74.0060,
    vertical_angle=10.0,
    horizontal_angle=90.0,
    back_view=False,
)
result = response['result']
print('Coordinates:', result.get('coordinates'))
print('Front keys :', list(result.get('front', {}).keys()))

In [ ]:
import base64, io
from PIL import Image
import matplotlib.pyplot as plt

def _decode(b64):
    if not b64: return None
    if b64.startswith('data:'): b64 = b64.split(',', 1)[1]
    return Image.open(io.BytesIO(base64.b64decode(b64)))

front = result.get('front', {})
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, key, title in [(axes[0], 'original_image', 'Street view'), (axes[1], 'segmented_image', 'Segmentation')]:
    img = _decode(front.get(key))
    if img is not None:
        ax.imshow(img)
    ax.set_title(title); ax.axis('off')
plt.tight_layout(); plt.show()

In [ ]:
segments = front.get('segments', {})
print('Class coverage:')
for cls, pct in sorted(segments.items(), key=lambda kv: kv[1], reverse=True):
    print(f'  {cls:>25}: {pct}')